# DATABASE

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display
from sql.queries import QUERIES
from utils import db_connection as db

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)


def show(key, rows=15):
    """Print the question title and SQL, then show the result table."""
    q = QUERIES[key]
    print(f"{key.upper()} [{q['level']}] {q['title']}")
    print("-" * 70)
    print(q["sql"], "\n")
    df = db.run_query(q["sql"])
    print(f"{len(df)} rows")
    display(df.head(rows))
    return df

# Cell 1: setup

In [1]:
import sys
sys.path.append("..")

from utils import db_connection as db
from sqlalchemy.exc import IntegrityError

db.init_db()
print("Database file:", db.get_engine().url)
db.table_counts()

Database file: sqlite:///C%3A/Users/avani/Documents/cricbuzz_livestats/data/cricbuzz.db


{'batting_performances': 0,
 'bowling_performances': 0,
 'matches': 0,
 'player_career_stats': 0,
 'players': 0,
 'series': 0,
 'teams': 0,
 'venues': 0}

# Cell 2: inspect the structure

In [2]:
tables = list(db.table_counts().keys())
for t in tables:
    cols = db.run_query(f'PRAGMA table_info("{t}")')
    print(f"\n== {t} ==")
    print(cols[["name", "type", "notnull", "pk"]].to_string(index=False))


== batting_performances ==
            name    type  notnull  pk
         perf_id INTEGER        0   1
        match_id INTEGER        1   0
       player_id INTEGER        1   0
         team_id INTEGER        0   0
      innings_no INTEGER        1   0
batting_position INTEGER        0   0
            runs INTEGER        1   0
     balls_faced INTEGER        1   0
           fours INTEGER        0   0
           sixes INTEGER        0   0
     strike_rate    REAL        0   0
          is_out INTEGER        0   0

== bowling_performances ==
         name    type  notnull  pk
      perf_id INTEGER        0   1
     match_id INTEGER        1   0
    player_id INTEGER        1   0
      team_id INTEGER        0   0
   innings_no INTEGER        1   0
        overs    REAL        1   0
      maidens INTEGER        0   0
runs_conceded INTEGER        1   0
      wickets INTEGER        1   0
 economy_rate    REAL        0   0

== matches ==
          name    type  notnull  pk
      match_id

# Cell 3: check the indexes

In [3]:
db.run_query(
    "SELECT name, tbl_name FROM sqlite_master "
    "WHERE type = 'index' AND name LIKE 'idx_%' ORDER BY tbl_name, name"
)

,name,tbl_name
0,idx_bat_match,batting_performances
1,idx_bat_player,batting_performances
2,idx_bowl_match,bowling_performances
3,idx_bowl_player,bowling_performances
4,idx_matches_date,matches
5,idx_matches_series,matches
6,idx_matches_venue,matches
7,idx_matches_winner,matches
8,idx_players_country,players
9,idx_players_role,players


# Cell 4: prove the constraints work

In [9]:
from sqlalchemy.exc import IntegrityError

CHILD_FIRST = ["batting_performances", "bowling_performances",
               "player_career_stats", "matches", "players",
               "venues", "series", "teams"]


def wipe_all():
    for table in CHILD_FIRST:
        db.execute(f"DELETE FROM {table}")


def expect_block(label, sql, params=None):
    try:
        db.execute(sql, params)
        print(f"NOT BLOCKED (bad!): {label}")
    except IntegrityError:
        print(f"BLOCKED (good):     {label}")


wipe_all()   # start from a clean slate every time

# Valid base rows
db.execute("INSERT INTO teams (team_id, team_name) VALUES (1, 'Test Team A')")
db.execute("INSERT INTO teams (team_id, team_name) VALUES (2, 'Test Team B')")
db.execute("INSERT INTO players (player_id, full_name) VALUES (1, 'Test Player')")
db.execute("INSERT INTO matches (match_id, team1_id, team2_id) VALUES (3, 1, 2)")

# 1) Foreign key: match pointing at teams that do not exist
expect_block("missing teams (FK)",
    "INSERT INTO matches (match_id, team1_id, team2_id) VALUES (10, 999, 998)")

# 2) CHECK: a team cannot play itself
expect_block("team1 == team2 (CHECK)",
    "INSERT INTO matches (match_id, team1_id, team2_id) VALUES (11, 1, 1)")

# 3) CHECK: negative runs
expect_block("negative runs (CHECK)",
    "INSERT INTO batting_performances (match_id, player_id, innings_no, runs) "
    "VALUES (3, 1, 1, -5)")

# 4) UNIQUE: same player twice in the same innings
db.execute("INSERT INTO batting_performances (match_id, player_id, innings_no, runs) "
           "VALUES (3, 1, 1, 50)")
expect_block("duplicate innings row (UNIQUE)",
    "INSERT INTO batting_performances (match_id, player_id, innings_no, runs) "
    "VALUES (3, 1, 1, 60)")

# 5) CHECK: invalid toss decision
expect_block("invalid toss_decision (CHECK)",
    "INSERT INTO matches (match_id, team1_id, team2_id, toss_decision) "
    "VALUES (12, 1, 2, 'sing')")

wipe_all()   # leave the database empty for Phase 9
print("\nRow counts after cleanup:", db.table_counts())

BLOCKED (good):     missing teams (FK)
BLOCKED (good):     team1 == team2 (CHECK)
BLOCKED (good):     negative runs (CHECK)
BLOCKED (good):     duplicate innings row (UNIQUE)
BLOCKED (good):     invalid toss_decision (CHECK)

Row counts after cleanup: {'batting_performances': 0, 'bowling_performances': 0, 'matches': 0, 'player_career_stats': 0, 'players': 0, 'series': 0, 'teams': 0, 'venues': 0}


# Cell 5: clean up the test rows

In [10]:
for table in ["batting_performances", "bowling_performances",
              "player_career_stats", "matches", "players",
              "venues", "series", "teams"]:
    db.execute(f"DELETE FROM {table}")
db.table_counts()

{'batting_performances': 0,
 'bowling_performances': 0,
 'matches': 0,
 'player_career_stats': 0,
 'players': 0,
 'series': 0,
 'teams': 0,
 'venues': 0}